> 在线讲义：[第 4 章 · 学习率调度与梯度裁剪：步子怎么排、梯度太大怎么办](https://wikieden.github.io/beyond-transformer/lessons/04-lr-schedule.html) · 本书《超越 Transformer》

# 第 4 章 · 学习率调度与梯度裁剪：步子怎么排、梯度太大怎么办

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/04-lr-schedule.html) · 仓库里 `beyond-transformer/notes/lessons/04-lr-schedule.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 调度 | 三段公式 | `lr_at(t)` | t = 550 → 5.5e-4 |
| 第二步 v̂ 的噪声 | 滑动平均 | `early_v_noise()` | 1 步 [0.061, 1.922] → 500 步 [0.952, 1.057] |
| 第三步 裁剪 | 整体缩放 | `clip(grads)` | [3, 4] → [0.6, 0.8] |

梯度裁剪（按全局 L2 范数缩放）。
只把讲义里的几个取值代进公式算一遍；a1 的 get_lr_cosine_schedule、gradient_clipping 两个 adapter 留给自己写。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
AMAX, AMIN, TW, TC = 1e-3, 1e-4, 100, 1000

**函数 `lr_at`**：a1 §4.4：t < Tw 线性升；Tw ≤ t ≤ Tc 余弦从 amax 降到 amin；t > Tc 停在 amin。

In [ ]:
def lr_at(t, amax=AMAX, amin=AMIN, tw=TW, tc=TC):
    """a1 §4.4：t < Tw 线性升；Tw ≤ t ≤ Tc 余弦从 amax 降到 amin；t > Tc 停在 amin。"""
    if t < tw:
        return t / tw * amax
    if t <= tc:
        return amin + 0.5 * (1 + math.cos((t - tw) / (tc - tw) * math.pi)) * (amax - amin)
    return amin

**函数 `early_v_noise`**：梯度 ~ N(0.1, 1)：前 k 步估计的 √v̂（Adam 用来除的分母）有多不稳。返回 k → (√v̂ 的 5% 分位, 95% 分位)。

In [ ]:
def early_v_noise(seed=0, n=2000):
    """梯度 ~ N(0.1, 1)：前 k 步估计的 √v̂（Adam 用来除的分母）有多不稳。返回 k → (√v̂ 的 5% 分位, 95% 分位)。"""
    rng = np.random.default_rng(seed)
    out = {}
    for k in (1, 5, 50, 500):
        vals = []
        for _ in range(n):
            g = rng.normal(0.1, 1.0, k)
            v = 0.0
            for i, gi in enumerate(g, 1):
                v = 0.999 * v + 0.001 * gi * gi
            vals.append(math.sqrt(v / (1 - 0.999**k)))
        out[k] = (float(np.quantile(vals, 0.05)), float(np.quantile(vals, 0.95)))
    return out

**函数 `clip`**：所有参数的梯度拼起来算一个 L2 范数；超过 max_norm 就整体乘 max_norm / (norm + eps)。

In [ ]:
def clip(grads, max_norm=1.0, eps=1e-6):
    """所有参数的梯度拼起来算一个 L2 范数；超过 max_norm 就整体乘 max_norm / (norm + eps)。"""
    norm = math.sqrt(sum(float((g * g).sum()) for g in grads))
    if norm <= max_norm:
        return grads, norm, 1.0
    s = max_norm / (norm + eps)
    return [g * s for g in grads], norm, s

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 warmup + cosine（αmax = 1e-3、αmin = 1e-4、Tw = 100、Tc = 1000）")
for t in (0, 50, 100, 325, 550, 775, 1000, 1200):
    print(f"  t = {t:4d}：lr = {lr_at(t):.6f}")
print("\n第二步 为什么要 warmup：前几步的 √v̂ 有多不准（梯度 ~ N(0.1, 1)，真值约 1.005）")
for k, (lo, hi) in early_v_noise().items():
    print(f"  只看了 {k:3d} 步：√v̂ 的 90% 范围 [{lo:.3f}, {hi:.3f}]")

In [ ]:
print("\n第三步 梯度裁剪")
for gs in ([np.array([3.0, 4.0])], [np.array([0.3, 0.4])], [np.array([3.0, 0.0]), np.array([[0.0, 4.0], [0.0, 0.0]])]):
    out, n, s = clip(gs)
    print(f"  梯度 {[g.tolist() for g in gs]}：范数 {n:.2f}，缩放 {s:.6f} → {[np.round(g, 4).tolist() for g in out]}")

## 3. 代码里才看得到的

- **边界的等号**：a1 写的是 t < Tw 走 warmup、Tw ≤ t ≤ Tc 走余弦，所以 t = Tw 时正好是 αmax，t = Tc 时正好是 αmin，两段接得上。
- **这里只是把公式代进去**：作业的 `get_lr_cosine_schedule` 和原地修改梯度的 `gradient_clipping` 两个 adapter 留给自己写。

## 4. 自己改一改

- 把 `lr_at` 的 αmin 改成 0，曲线的后半段怎么变？
- 把 `early_v_noise` 里的梯度均值从 0.1 改成 2，第 1 步的范围还那么宽吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 先热身、再全速、最后慢慢刹：线性升到峰，余弦降到底，之后停在最低。
> 退火走一半，学习率在中点。
> 开头梯度乱、v̂ 不准，warmup 走小步。
> 所有梯度拼一条，超长就按比例缩，方向不变；裁前范数记日志。